# KADIMA fast-entry review — 6 October 2026
## tl;dr
PR #28 cannot be replayed from the available CSV. It contains 13 suggestions and four closed Manual records, with no decision-time candles, option paths or BUY confirmations. Nine suggestions have SL as their first recorded level event; four have T1 first. These are observations, not broker-confirmed wins/losses.
## Context & Methods
Primary source: [audit CSV](https://github.com/manchalakusumakar-spec/kadima-audit/blob/a045bb431a2ade6399a703d9e77d405dd0bec16b/audit/2026-10-05.csv), source Git blob `e83aaa81755991840fdc4fbd39b20e51a6fe8f76`. Fetched 6 October 2026, approximately 10:47–10:55 IST. The included copy normalizes line endings only.
### Key Assumptions
Each exported suggestion row stays separate because immutable IDs are missing. Event ordering uses the supplied timestamps. Missing events are not assumed impossible. A common 15-minute window is a sensitivity analysis, not a reconstruction of missing planned horizons. Engine P&L is unverified against broker fills. No chart is needed for this small evidence-readiness check; exact tables preserve event order.
## Data
Run from the repository root or the analysis directory; only Python's standard library is needed for the cells. No network or broker calls occur.

In [1]:
import csv, json, hashlib
from pathlib import Path
from collections import Counter
from datetime import datetime
from decimal import Decimal
path = Path('analysis/evidence/audit_2026-10-05.csv')
if not path.exists():
    path = Path('evidence/audit_2026-10-05.csv')
raw = path.read_bytes()
rows = list(csv.DictReader(raw.decode().splitlines()))
suggestions = [r for r in rows if r['Origin'] == 'SUGGESTION']
closed = [r for r in rows if r['Origin'] == 'PERSISTED_SIGNAL' and r['Status'] == 'CLOSED']
assert len(rows) == 17 and len(suggestions) == 13 and len(closed) == 4
assert len({tuple(r.values()) for r in rows}) == 17
assert all(datetime.fromisoformat(r['Suggested at IST']).utcoffset().total_seconds() == 19800 for r in rows)
print(json.dumps(dict(rows=len(rows), columns=len(rows[0]), suggestions=len(suggestions), closed_manual_records=len(closed), normalized_csv_sha256=hashlib.sha256(raw).hexdigest(), quote_status=dict(Counter(r['Quote status'] for r in rows))), indent=2))

{
  "rows": 17,
  "columns": 41,
  "suggestions": 13,
  "closed_manual_records": 4,
  "normalized_csv_sha256": "71ceb7d67122aa7c26c6b04b8fc6bc76c0603b5a9127c9168e7c0e25cd536958",
  "quote_status": {
    "STALE": 17
  }
}


## Results
### First recorded events and original horizons

In [2]:
def events(row):
    start = datetime.fromisoformat(row['Suggested at IST'])
    found = []
    for field, label in [('T1 observed IST','T1'), ('T2 observed IST','T2'), ('SL observed IST','SL')]:
        if row[field]:
            elapsed = (datetime.fromisoformat(row[field])-start).total_seconds()
            assert elapsed >= 0
            found.append((elapsed,label))
    return sorted(found)

def first_event(row, seconds=None):
    found = [e for e in events(row) if seconds is None or e[0] <= seconds]
    if not found:
        return 'NO_RECORDED_EVENT'
    if sum(e[0] == found[0][0] for e in found) > 1:
        return 'AMBIGUOUS'
    return found[0][1]

summary = dict(first_recorded=dict(Counter(first_event(r) for r in suggestions)),
    any_recorded_sl=sum(bool(r['SL observed IST']) for r in suggestions),
    sl_within_four_minutes=sum(any(t <= 240 and label == 'SL' for t,label in events(r)) for r in suggestions),
    missing_suggestion_ids=sum(not r['Signal ID'] for r in suggestions),
    missing_original_horizon=sum(not r['Proposed maximum hold minutes'] for r in suggestions))
assert summary['first_recorded'] == {'SL':9,'T1':4}
assert summary['any_recorded_sl'] == 10
assert summary['missing_suggestion_ids'] == 13
assert summary['missing_original_horizon'] == 8
print(json.dumps(summary, indent=2))
known = [r for r in suggestions if r['Proposed maximum hold minutes']]
print('Recorded-horizon cases:',len(known),dict(Counter(first_event(r,float(r['Proposed maximum hold minutes'])*60) for r in known)))
print('Hypothetical 15-minute sensitivity:',dict(Counter(first_event(r,900) for r in suggestions)))

{
  "first_recorded": {
    "SL": 9,
    "T1": 4
  },
  "any_recorded_sl": 10,
  "sl_within_four_minutes": 5,
  "missing_suggestion_ids": 13,
  "missing_original_horizon": 8
}
Recorded-horizon cases: 5 {'SL': 4, 'T1': 1}
Hypothetical 15-minute sensitivity: {'SL': 8, 'NO_RECORDED_EVENT': 3, 'T1': 2}


### Event-time audit
Repeated contracts are intentionally retained. Minutes are measured from the exported suggestion timestamp.

In [3]:
print('Contract | Suggested IST | First event | Event minutes | Original horizon')
for r in suggestions:
    timing = ', '.join(f'{label} {t/60:.2f}' for t,label in events(r))
    print(f"{r['Strike / contract']} | {r['Suggested at IST'][11:19]} | {first_event(r)} | {timing} | {r['Proposed maximum hold minutes'] or 'UNAVAILABLE'}")
bank = next(r for r in suggestions if r['Strike / contract']=='BANKNIFTY27OCT2654700CE')
assert first_event(bank) == 'T1' and bank['Observed outcome'] == 'SL_OBSERVED'
gail = next(r for r in suggestions if r['Strike / contract']=='GAIL27OCT26170PE')
assert first_event(gail,900) == 'T1'
repeated = [r for r in suggestions if r['Strike / contract']=='NIFTY06OCT2622500CE']
print('Repeated-contract separation seconds:',abs((datetime.fromisoformat(repeated[0]['Suggested at IST'])-datetime.fromisoformat(repeated[1]['Suggested at IST'])).total_seconds()))

Contract | Suggested IST | First event | Event minutes | Original horizon
AUBANK27OCT261030CE | 13:56:54 | SL | SL 11.15 | 15.0
MIDCPNIFTY27OCT2613700CE | 12:22:37 | SL | SL 10.29 | UNAVAILABLE
BANKNIFTY27OCT2654700CE | 12:21:13 | T1 | T1 79.53, SL 195.66 | UNAVAILABLE
NIFTY06OCT2622500PE | 11:40:01 | T1 | T1 10.20, T2 20.32 | UNAVAILABLE
NIFTY06OCT2622500CE | 11:31:01 | SL | SL 7.21 | UNAVAILABLE
NIFTY06OCT2622500CE | 11:30:29 | SL | SL 15.15 | UNAVAILABLE
VOLTAS27OCT261080CE | 11:01:16 | SL | SL 2.25 | 15.0
BANKNIFTY27OCT2655100PE | 10:40:06 | SL | SL 3.45 | UNAVAILABLE
GAIL27OCT26170PE | 10:24:41 | T1 | T1 2.25 | 15.0
SENSEX26O0872400PE | 10:06:06 | T1 | T1 19.24, T2 48.93 | UNAVAILABLE
SENSEX26O0872600CE | 09:51:04 | SL | SL 2.28 | UNAVAILABLE
MOTILALOFS27OCT261020CE | 09:47:13 | SL | SL 2.57 | 15.0
ICICIGI27OCT261600CE | 09:34:15 | SL | SL 3.47 | 15.0
Repeated-contract separation seconds: 31.585007


### Closed Manual rows: missing BUY evidence
Do not substitute a suggestion/adoption timestamp for an actual entry fill.

In [4]:
reported_pnl = sum(Decimal(r['Realized P&L before fees']) for r in closed)
assert reported_pnl == Decimal('-387.5')
assert all(r['Trade type']=='Manual' and r['Exit reason']=='underlying_lost_vwap' for r in closed)
assert all(not r['First BUY confirmation IST'] and not r['Actual entry'] and Decimal(r['Bought quantity'])==0 for r in closed)
print('Contract | Signal ID | Reported gross P&L INR | Signal-to-SELL seconds | Actual hold')
for r in closed:
    delta=(datetime.fromisoformat(r['Last SELL confirmation IST'])-datetime.fromisoformat(r['Suggested at IST'])).total_seconds()
    print(f"{r['Strike / contract']} | {r['Signal ID']} | {r['Realized P&L before fees']} | {delta:.3f} | UNAVAILABLE")
print('Total engine-reported P&L before fees:',str(reported_pnl),'INR; broker reconciliation and net P&L unavailable')

Contract | Signal ID | Reported gross P&L INR | Signal-to-SELL seconds | Actual hold
NIFTY06OCT2622350PE | 226 | -45.5 | 8.258 | UNAVAILABLE
SENSEX26O0871800PE | 225 | -112.0 | 9.380 | UNAVAILABLE
SENSEX26O0872100CE | 224 | -80.0 | 8.307 | UNAVAILABLE
VOLTAS27OCT261080CE | 223 | -150.0 | 8.271 | UNAVAILABLE
Total engine-reported P&L before fees: -387.5 INR; broker reconciliation and net P&L unavailable


## Takeaways
- Zero of 13 suggestions can be assessed by the new candle filters from this export; no simulated pass/fail or profit is invented.
- The one-day sample and missing independent IDs cannot calibrate thresholds or prove a score band works better.
- Four Manual records cite VWAP exits. The inspected source permits a fresh adverse VWAP comparison to trigger an exit, but historical deployment, adoption chronology and market context are missing. This is a mechanism to investigate, not a verified root cause.
- Obtain closed 1m/5m candles, candidate payloads, option bid/ask and volume paths, contract identity, complete broker fills and costs, and deployed source/settings. Then replay the frozen baseline and patch with identical capital and cost assumptions on held-out days.

Validation: calculations executed sequentially using plain Python and saved below. No Jupyter kernel, nbformat library or notebook viewer was available, so notebook-viewer rendering remains unverified. See `docs/fast_entry_evidence_20261006.md` for the full source/readiness review.